# Day 1 — Document Ingestion
### MedFlow: Evidence-Grounded Medical RAG for Thyroid Diseases

**Objective:**
1. Ingest real clinical guideline PDFs and patient brochures from `data/`
2. Clean noisy medical text and preserve metadata
3. Compare Naive Fixed-Size chunking vs. Section-Aware chunking
4. Generate embeddings with `BAAI/bge-small-en-v1.5`
5. Build and persist a vector index in ChromaDB


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from day1.ingest import load_pdfs, naive_chunk_documents, section_aware_chunk_documents, build_index
import config

print("Data directory:", os.path.abspath("../data"))
pages = load_pdfs("../data")
print(f"Total pages loaded: {len(pages)}")


In [ ]:
# Compare Chunking Strategies
naive_chunks = naive_chunk_documents(pages, chunk_size=500, chunk_overlap=50)
section_chunks = section_aware_chunk_documents(pages, chunk_size=550, chunk_overlap=70)

print(f"Naive Chunks: {len(naive_chunks)}")
print(f"Section-Aware Chunks: {len(section_chunks)}")
print("\nSample Section Chunk:")
print("Metadata:", section_chunks[0].metadata)
print("Content:", section_chunks[0].page_content[:300])


In [ ]:
# Build Vector Store
vectordb = build_index(section_chunks, persist_dir="../chroma_db")
print("ChromaDB vector store persisted successfully.")


In [ ]:
# Baseline Query Execution
query = "How is hypothyroidism diagnosed?"
results = vectordb.similarity_search_with_score(query, k=3)

print(f"QUERY: {query}\n")
for idx, (doc, dist) in enumerate(results, 1):
    sim = max(0.0, 1.0 - (dist / 2.0))
    print(f"[{idx}] {doc.metadata.get('document_name')} (Page {doc.metadata.get('page_number')}) | Sim: {sim:.4f}")
    print(f"Passage: {doc.page_content[:200]}...\n")
